# Square-system finite-time Lyapunov gap: T=10, ten samples

Hard walls, slab-only measurements, alpha1=1, alpha2=30, nshell=1;
L=Nx=Ny=20,24,28,32,36,40,44. Each case starts maximally mixed with the
canonical Born-conditioned exterior preparation, then runs ten physical
raster-y cycles with perfect correction. Production requires A100 40-GB-class.

At cycle ten only, save each active-slab occupation spectrum. Define
$g_{\mathrm{mod},\xi}=\min_j|\log[(1-\nu_{\xi,j})/\nu_{\xi,j}]|$ and
$\Delta_\xi=g_{\mathrm{mod},\xi}/(2T)=g_{\mathrm{mod},\xi}/20$.
Average gaps over ten trajectories; uncertainty is ordinary SEM, not bootstrap.
This is a finite-time diagnostic, not proof of a converged scaling exponent.
Both wall separation and circumference increase with L.

One ten-sample batch per size; two five-sample result shards. Rolling covariance
and RNG checkpoints every five cycles. No intermediate spectral observations,
contours, eigenvectors, or covariance histories. Never run two writers concurrently.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

REPORT_ONLY = False
MAX_NEW_EXECUTION_BATCHES = None  # use 1 for the first L=44 batch
RUN_ANALYSIS = False  # requires all 14 result shards
# Complete configuration: scientific changes require a new revision.
CONFIG = {'sampling_revision': 'square_hard_wall_gap_l20-44_s10_t10_v1',
 'root_seed': 2026092725,
 'sizes': [20, 24, 28, 32, 36, 40, 44],
 'samples_per_case': 10,
 'cycles': 10,
 'execution_batch_size': 10,
 'result_shard_size': 5,
 'segment_cycles': 5,
 'DW': True,
 'dw_truncation': True,
 'meas_slab_only': True,
 'triv_region_local_mode': False,
 'alpha_1': 1.0,
 'alpha_2': 30.0,
 'nshell': 1,
 'trial_orbitals': 'X',
 'filling_fraction': 0.5,
 'init_mode': 'maxmix',
 'n_a': 0.5,
 'sequence': 'raster_y',
 'perfect_correction': True,
 'postselect': False,
 'state_representation': 'covariance',
 'dtype': 'complex128',
 'device': 'cuda:0',
 'backend': 'local',
 'cap_tolerance': 1e-09,
 'initial_purity_tolerance': 0.50000001,
 'gpu_allocator_limit_bytes': 35000000000,
 'spectrum_cycles': [10],
 'observer_sample_chunk': 1,
 'covariance_spectral_clip': False}
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/25_square_hard_wall_gap_pilot')
LOCAL_BUNDLE_DIR = Path('/content/25_square_hard_wall_gap_pilot')
SCRATCH_ROOT = Path('/content/square_gap_scratch')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs') / CONFIG['sampling_revision']


## Stage locally and run/resume

The largest geometry runs first. The runner prints checkpoint status, measured cycle/checkpoint times and GPU memory. A failed write leaves the task incomplete; cycle-10 checkpoints survive until both result shards verify.

In [ ]:
import codecs
import json
import shutil
import subprocess
import sys

LOCAL_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
for relative in ['run_campaign.py', 'endpoint_spectrum.py', 'src/classA_U1FGTN_gpu.py', 'src/occupied_frame_gpu.py', 'analyze_campaign.py']:
    destination = LOCAL_BUNDLE_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BUNDLE_DRIVE_DIR / relative, destination)
config_path = LOCAL_BUNDLE_DIR / 'resolved_config.json'
config_path.write_text(json.dumps(CONFIG, indent=2))

def run_streamed(command):
    print('[launch]', ' '.join(command), flush=True)
    decoder = codecs.getincrementaldecoder('utf-8')('replace')
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0) as process:
        while True:
            chunk = process.stdout.read(4096)
            if not chunk:
                break
            sys.stdout.write(decoder.decode(chunk)); sys.stdout.flush()
        sys.stdout.write(decoder.decode(b'', final=True)); sys.stdout.flush()
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)

command = [sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'run_campaign.py'),
           '--config', str(config_path), '--output-root', str(OUTPUT_ROOT),
           '--scratch-root', str(SCRATCH_ROOT)]
if REPORT_ONLY:
    command.append('--report-only')
if MAX_NEW_EXECUTION_BATCHES is not None:
    command.extend(['--max-new-execution-batches', str(MAX_NEW_EXECUTION_BATCHES)])
run_streamed(command)


## Optional endpoint analysis

Mean of sample-resolved gaps with ordinary SEM; raw modular gap and normalized Lyapunov half gap are plotted separately. No fitted exponent.

In [ ]:
if RUN_ANALYSIS and not REPORT_ONLY:
    run_streamed([sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'analyze_campaign.py'),
                  '--output-root', str(OUTPUT_ROOT)])
else:
    print('Analysis skipped.')


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
